# 07 - Modelo hÃ­brido con LightFM

LightFM es una librerÃ­a para construir recomendadores que combinan filtrado colaborativo y contenido. En lugar de usar solo similitud entre pelÃ­culas o solo patrones de usuarios, aprende representaciones latentes de usuarios, pelÃ­culas y features de los Ã­tems, como gÃ©neros, dÃ©cadas o popularidad.

Este notebook es un experimento independiente. Entrena un modelo LightFM hÃ­brido usando las interacciones positivas de MovieLens y aÃ±ade mis ratings reales de Trakt como un usuario adicional llamado `trakt_user`.

Este experimento no sustituye todavÃ­a al recomendador explicable actual. Sirve para probar si LightFM puede aportar una seÃ±al colaborativa y de contenido que mÃ¡s adelante pueda integrarse en un recomendador hÃ­brido final.

## 1. ComprobaciÃ³n del entorno

In [1]:
import re
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import scipy

from lightfm import LightFM
from lightfm.data import Dataset

print("Python executable:", sys.executable)
print("numpy version:", np.__version__)
print("scipy version:", scipy.__version__)
print("LightFM OK")

c:\Users\alexo\miniconda3\envs\tfg-lightfm\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


Python executable: c:\Users\alexo\miniconda3\envs\tfg-lightfm\python.exe
numpy version: 1.26.4
scipy version: 1.15.2
LightFM OK


## 2. DetecciÃ³n de rutas

In [2]:
def find_base_dir(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se ha encontrado una carpeta 'data' en la ruta actual ni en sus padres.")


base_dir = find_base_dir()

paths = {
    "ratings": base_dir / "data" / "raw" / "ratings.csv",
    "movies": base_dir / "data" / "processed" / "movies_clean.csv",
    "trakt_ratings": base_dir / "data" / "processed" / "trakt_ratings_mapped.csv",
    "trakt_watched": base_dir / "data" / "processed" / "trakt_watched_mapped.csv",
}

required_path_names = {"ratings", "movies", "trakt_ratings"}

print("base_dir:", base_dir)
for name, path in paths.items():
    exists = path.exists()
    optional_note = "" if name in required_path_names else " (opcional)"
    print(f"{name}: {path} -> {'OK' if exists else 'NO ENCONTRADO'}{optional_note}")
    if not exists and name in required_path_names:
        raise FileNotFoundError(f"Falta el archivo requerido: {path}")

base_dir: C:\Users\alexo\Desktop\TFG
ratings: C:\Users\alexo\Desktop\TFG\data\raw\ratings.csv -> OK
movies: C:\Users\alexo\Desktop\TFG\data\processed\movies_clean.csv -> OK
trakt_ratings: C:\Users\alexo\Desktop\TFG\data\processed\trakt_ratings_mapped.csv -> OK
trakt_watched: C:\Users\alexo\Desktop\TFG\data\processed\trakt_watched_mapped.csv -> OK (opcional)


## 3. Carga de datos

In [3]:
movies_lightfm_path = base_dir / "data" / "processed" / "movies_with_tags_lightfm.csv"
movies_source_path = movies_lightfm_path if movies_lightfm_path.exists() else paths["movies"]

ratings = pd.read_csv(paths["ratings"])
movies = pd.read_csv(movies_source_path)
trakt_ratings = pd.read_csv(paths["trakt_ratings"])
trakt_watched = pd.read_csv(paths["trakt_watched"]) if paths["trakt_watched"].exists() else pd.DataFrame()

print("movies source:", movies_source_path)
if movies_source_path == movies_lightfm_path:
    print("Usando movies_with_tags_lightfm.csv para LightFM.")
else:
    print("Aviso: no existe movies_with_tags_lightfm.csv; se usara movies_clean.csv sin tags preprocesados.")

def print_df_summary(name, df, main_columns=None):
    print(f"\n{name}")
    print("shape:", df.shape)
    print("columnas:", list(df.columns))
    if main_columns:
        available = [col for col in main_columns if col in df.columns]
        if available:
            display(df[available].head())

print_df_summary("ratings", ratings, ["userId", "movieId", "rating", "timestamp"])
print_df_summary("movies", movies, ["movieId", "title", "year", "genres", "rating_mean", "rating_count", "tags_features_en"])
print_df_summary("trakt_ratings", trakt_ratings, ["title", "year", "movieId", "user_rating_normalized", "match_source"])
print_df_summary("trakt_watched", trakt_watched, ["title", "year", "movieId", "plays", "last_watched_at", "match_source"])

movies source: C:\Users\alexo\Desktop\TFG\data\processed\movies_with_tags_lightfm.csv
Usando movies_with_tags_lightfm.csv para LightFM.

ratings
shape: (32000204, 4)
columnas: ['userId', 'movieId', 'rating', 'timestamp']


,userId,movieId,rating,timestamp
0,1,17,4.0,944249077
1,1,25,1.0,944250228
2,1,29,2.0,943230976
3,1,30,5.0,944249077
4,1,32,5.0,943228858



movies
shape: (87585, 27)
columnas: ['movieId', 'title', 'genres', 'year', 'title_clean', 'rating_mean', 'rating_count', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'tags_features_en']


,movieId,title,year,genres,rating_mean,rating_count,tags_features_en
0,1,Toy Story (1995),1995.0,Adventure|Animation|Children|Comedy|Fantasy,3.897438,68997,violence|comedy|friendship|funny|clv|cult_film...
1,2,Jumanji (1995),1995.0,Adventure|Children|Fantasy,3.275758,28904,violence|comedy|friendship|based_on_a_book|pol...
2,3,Grumpier Old Men (1995),1995.0,Comedy|Romance,3.139447,13134,funny|clv|sequel|wedding|duringcreditsstinger|...
3,4,Waiting to Exhale (1995),1995.0,Comedy|Drama|Romance,2.845331,2806,revenge|clv|based_on_novel_or_book|divorce|sin...
4,5,Father of the Bride Part II (1995),1995.0,Comedy,3.059602,13154,comedy|clv|family|sequel|fantasy|remake|parent...



trakt_ratings
shape: (147, 23)
columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 'movieId_tmdb', 'movieId_imdb', 'movieId', 'match_source', 'title_ml', 'title_clean_ml', 'year_ml', 'genres_ml', 'rating_mean_ml', 'rating_count_ml']


,title,year,movieId,user_rating_normalized,match_source
0,Sundays,2025,NaN,3.0,unmapped
1,Cinema Paradiso,1988,1172.0,4.0,tmdb
2,Dora and the Lost City of Gold,2019,203224.0,3.0,tmdb
3,The Housemaid,2025,NaN,2.0,unmapped
4,Marty Supreme,2025,NaN,5.0,unmapped



trakt_watched
shape: (256, 23)
columnas: ['title', 'year', 'trakt_id', 'slug', 'imdb_id', 'tmdb_id', 'user_rating_trakt', 'user_rating_normalized', 'rated_at', 'plays', 'last_watched_at', 'imdbId_norm', 'tmdbId_norm', 'movieId_tmdb', 'movieId_imdb', 'movieId', 'match_source', 'title_ml', 'title_clean_ml', 'year_ml', 'genres_ml', 'rating_mean_ml', 'rating_count_ml']


,title,year,movieId,plays,last_watched_at,match_source
0,Hoppers,2026,NaN,4,2026-03-29T00:54:00.000Z,unmapped
1,Ted,2012,95441.0,3,2026-04-26T14:11:00.000Z,tmdb
2,Coraline,2009,66097.0,2,2026-01-30T18:46:00.000Z,tmdb
3,Whiplash,2014,112552.0,2,2026-04-29T23:24:00.000Z,tmdb
4,The Bad Guys,2022,272407.0,2,2026-02-08T23:59:00.000Z,tmdb


## 4. Preparar interacciones MovieLens

In [4]:
required_ratings_cols = {"userId", "movieId", "rating"}
missing_ratings_cols = required_ratings_cols - set(ratings.columns)
if missing_ratings_cols:
    raise ValueError(f"Faltan columnas en ratings.csv: {missing_ratings_cols}")

ml_positive = ratings.loc[ratings["rating"] >= 4.0, ["userId", "movieId", "rating"]].copy()
ml_positive = ml_positive.dropna(subset=["userId", "movieId", "rating"])
ml_positive["user_id"] = "ml_" + ml_positive["userId"].astype(str)
ml_positive["movieId"] = pd.to_numeric(ml_positive["movieId"], errors="coerce")
ml_positive = ml_positive.dropna(subset=["movieId"])
ml_positive["movieId"] = ml_positive["movieId"].astype("int64")

ml_interactions = ml_positive[["user_id", "movieId", "rating"]].copy()

print("Interacciones positivas MovieLens (rating >= 4.0):", len(ml_interactions))
print("Usuarios MovieLens con interacciones positivas:", ml_interactions["user_id"].nunique())
print("PelÃ­culas MovieLens con interacciones positivas:", ml_interactions["movieId"].nunique())
display(ml_interactions.head())

Interacciones positivas MovieLens (rating >= 4.0): 15938231
Usuarios MovieLens con interacciones positivas: 200726
PelÃ­culas MovieLens con interacciones positivas: 55174


,user_id,movieId,rating
0,ml_1,17,4.0
3,ml_1,30,5.0
4,ml_1,32,5.0
7,ml_1,80,5.0
9,ml_1,111,5.0


## 5. Preparar interacciones Trakt

In [5]:
required_trakt_cols = {"movieId", "user_rating_normalized"}
missing_trakt_cols = required_trakt_cols - set(trakt_ratings.columns)
if missing_trakt_cols:
    raise ValueError(f"Faltan columnas en trakt_ratings_mapped.csv: {missing_trakt_cols}")

trakt_ratings_work = trakt_ratings.copy()
trakt_ratings_work["movieId"] = pd.to_numeric(trakt_ratings_work["movieId"], errors="coerce")
trakt_ratings_work["user_rating_normalized"] = pd.to_numeric(
    trakt_ratings_work["user_rating_normalized"], errors="coerce"
)

trakt_mapped_ratings = trakt_ratings_work.dropna(subset=["movieId", "user_rating_normalized"]).copy()
trakt_mapped_ratings["movieId"] = trakt_mapped_ratings["movieId"].astype("int64")

trakt_positive = trakt_mapped_ratings.loc[
    trakt_mapped_ratings["user_rating_normalized"] >= 4.0,
    ["movieId", "user_rating_normalized"],
].copy()
trakt_positive["user_id"] = "trakt_user"
trakt_positive = trakt_positive.rename(columns={"user_rating_normalized": "rating"})
trakt_interactions = trakt_positive[["user_id", "movieId", "rating"]].copy()

rated_movie_ids = set(trakt_mapped_ratings["movieId"].astype("int64"))

print("Ratings totales en Trakt:", len(trakt_ratings_work))
print("Ratings de Trakt mapeados a MovieLens:", len(trakt_mapped_ratings))
print("Ratings positivos de Trakt (>= 4.0):", len(trakt_interactions))
print("PelÃ­culas ya valoradas en Trakt:", len(rated_movie_ids))
display(trakt_interactions.head())

Ratings totales en Trakt: 147
Ratings de Trakt mapeados a MovieLens: 123
Ratings positivos de Trakt (>= 4.0): 47
PelÃ­culas ya valoradas en Trakt: 123


,user_id,movieId,rating
1,trakt_user,1172,4.0
5,trakt_user,6016,4.0
11,trakt_user,2291,4.0
15,trakt_user,115569,4.0
17,trakt_user,104913,4.0


## 6. Unir interacciones

In [6]:
all_interactions = pd.concat([ml_interactions, trakt_interactions], ignore_index=True)
all_interactions["movieId"] = pd.to_numeric(all_interactions["movieId"], errors="coerce")
all_interactions["rating"] = pd.to_numeric(all_interactions["rating"], errors="coerce")
all_interactions = all_interactions.dropna(subset=["user_id", "movieId", "rating"])
all_interactions["movieId"] = all_interactions["movieId"].astype("int64")

# Si hubiera duplicados usuario-pelÃ­cula, conservamos la valoraciÃ³n positiva mÃ¡s alta como peso.
all_interactions = (
    all_interactions.groupby(["user_id", "movieId"], as_index=False)["rating"]
    .max()
    .sort_values(["user_id", "movieId"])
)

known_movie_ids = set(pd.to_numeric(movies["movieId"], errors="coerce").dropna().astype("int64"))
before_filter = len(all_interactions)
all_interactions = all_interactions.loc[all_interactions["movieId"].isin(known_movie_ids)].copy()
discarded_interactions = before_filter - len(all_interactions)

print("Usuarios totales:", all_interactions["user_id"].nunique())
print("PelÃ­culas totales con interacciones:", all_interactions["movieId"].nunique())
print("Interacciones positivas totales:", len(all_interactions))
print("Interacciones descartadas por no estar en movies_clean:", discarded_interactions)
print("Interacciones de trakt_user:", (all_interactions["user_id"] == "trakt_user").sum())
display(all_interactions.head())

Usuarios totales: 200727
PelÃ­culas totales con interacciones: 55174
Interacciones positivas totales: 15938278
Interacciones descartadas por no estar en movies_clean: 0
Interacciones de trakt_user: 47


,user_id,movieId,rating
0,ml_1,17,4.0
1,ml_1,30,5.0
2,ml_1,32,5.0
3,ml_1,80,5.0
4,ml_1,111,5.0


## 7. Crear features de pelÃ­culas

In [7]:
if "movieId" not in movies.columns:
    raise ValueError("movies_with_tags_lightfm.csv o movies_clean.csv debe contener la columna movieId")

movies_before_clean = len(movies)
movies_model = movies.copy()

if "genres" not in movies_model.columns:
    raise ValueError("movies_clean.csv debe contener la columna genres")
if "year" not in movies_model.columns:
    raise ValueError("movies_clean.csv debe contener la columna year")

movies_model["movieId"] = pd.to_numeric(movies_model["movieId"], errors="coerce")
movies_model["year"] = pd.to_numeric(movies_model["year"], errors="coerce")
movies_model["genres"] = movies_model["genres"].fillna("").astype(str).str.strip()
movies_model = movies_model.dropna(subset=["movieId", "year"]).copy()
movies_model["movieId"] = movies_model["movieId"].astype("int64")
movies_model = movies_model.loc[
    movies_model["genres"].ne("")
    & movies_model["genres"].ne("(no genres listed)")
].copy()
movies_model = movies_model.drop_duplicates(subset=["movieId"]).copy()
movies_after_clean = len(movies_model)

def popularity_bucket(value):
    if pd.isna(value):
        return None
    if value < popularity_q1:
        return "popularity_low"
    if value < popularity_q2:
        return "popularity_medium"
    return "popularity_high"

def normalize_feature_token(value):
    token = re.sub(r"[^a-z0-9]+", "_", str(value).strip().lower())
    token = re.sub(r"_+", "_", token).strip("_")
    return token

def extract_tags_from_value(tag_value):
    if pd.isna(tag_value):
        return []

    tag_tokens = re.split(r"[|,;/\\n]+", str(tag_value))
    cleaned_tags = []
    for raw_tag in tag_tokens:
        raw_tag = raw_tag.strip().lower()
        if not raw_tag:
            continue
        raw_tag = raw_tag.replace("&", " and ")
        normalized = normalize_feature_token(raw_tag)
        if not normalized:
            continue
        if normalized in BLOCKED_TAG_FEATURES:
            continue
        if normalized.isdigit() or len(normalized) < 3:
            continue
        cleaned_tags.append(normalized)
    return cleaned_tags

USE_TAG_FEATURES = True
MAX_TAG_FEATURES = 100
BLOCKED_TAG_FEATURES = {
    "oscar",
    "nominee",
    "award",
    "awards",
    "best",
    "great",
    "good",
    "bad",
    "movie",
    "film",
    "films",
    "cinema",
    "classic",
    "ending",
    "nudity",
    "shot",
    "death",
    "story",
    "acting",
    "music",
    "reference",
    "relationship",
    "man",
    "woman",
    "male",
    "female",
    "boy",
    "girl",
    "character",
    "title",
    "dvd",
    "blu_ray",
    "netflix",
    "watched",
    "seen",
    "based",
    "true",
    "book",
    "adaptation",
    "adapted",
    "year",
    "old",
    "young",
    "girl",
    "boy",
    "woman",
    "man",
}

has_rating_count = "rating_count" in movies_model.columns
if has_rating_count:
    movies_model["rating_count"] = pd.to_numeric(movies_model["rating_count"], errors="coerce")
    popularity_q1 = movies_model["rating_count"].quantile(0.33)
    popularity_q2 = movies_model["rating_count"].quantile(0.66)

tag_source_column = next((column for column in ["tags_features_en", "tags_combined_en"] if column in movies_model.columns), None)
movie_tag_candidates = {}
selected_tag_features = set()
tag_counter = Counter()
print("USE_TAG_FEATURES:", USE_TAG_FEATURES)
print("tags_features_en existe:", "tags_features_en" in movies_model.columns)
print("Columna usada para tags:", tag_source_column if tag_source_column is not None else "ninguna")
if tag_source_column is None:
    print("LightFM esta entrenando sin tags; solo usa generos, decada y popularidad.")
else:
    for row in movies_model.itertuples(index=False):
        movie_id = int(getattr(row, "movieId"))
        tags = extract_tags_from_value(getattr(row, tag_source_column))
        movie_tag_candidates[movie_id] = tags
        tag_counter.update(set(tags))
    print("Tags candidatos detectados:", len(tag_counter))
    if USE_TAG_FEATURES:
        selected_tag_features = {tag for tag, _ in tag_counter.most_common(MAX_TAG_FEATURES)}
    else:
        print("USE_TAG_FEATURES=False; LightFM esta entrenando sin tags; solo usa generos, decada y popularidad.")
    print("Tags usados como features:", len(selected_tag_features))
    print("Primeros tags usados:", sorted(selected_tag_features)[:100])

movie_features = {}
genre_feature_names = set()
decade_feature_names = set()
popularity_feature_names = set()
tag_feature_names = set()
all_item_features = set()

for row in movies_model.itertuples(index=False):
    movie_id = int(getattr(row, "movieId"))
    features = []

    genres_value = getattr(row, "genres", "") if "genres" in movies_model.columns else ""
    if pd.notna(genres_value):
        for genre in str(genres_value).split("|"):
            genre = genre.strip()
            if genre and genre != "(no genres listed)":
                feature = f"genre_{genre}"
                features.append(feature)
                genre_feature_names.add(feature)

    year_value = getattr(row, "year")
    if pd.notna(year_value):
        decade = int(float(year_value) // 10 * 10)
        feature = f"decade_{decade}"
        features.append(feature)
        decade_feature_names.add(feature)

    if has_rating_count:
        bucket = popularity_bucket(getattr(row, "rating_count"))
        if bucket:
            features.append(bucket)
            popularity_feature_names.add(bucket)

    if selected_tag_features and movie_id in movie_tag_candidates:
        for tag in movie_tag_candidates[movie_id]:
            if tag in selected_tag_features:
                feature = f"tag_{tag}"
                features.append(feature)
                tag_feature_names.add(feature)

    movie_features[movie_id] = list(dict.fromkeys(features))
    all_item_features.update(movie_features[movie_id])

print("Peliculas con features:", len(movie_features))
print("Numero de features unicas:", len(all_item_features))
print("Peliculas antes de limpiar:", movies_before_clean)
print("Peliculas despues de limpiar:", movies_after_clean)
print("Numero de features de genero:", len(genre_feature_names))
print("Numero de features de decada:", len(decade_feature_names))
print("Numero de features de popularidad:", len(popularity_feature_names))
print("Numero de tags usados como features:", len(tag_feature_names))
print("Numero total de item_features:", len(all_item_features))
print("Ejemplos de features:")
for movie_id, features in list(movie_features.items())[:5]:
    title = movies_model.loc[movies_model["movieId"] == movie_id, "title"].iloc[0] if "title" in movies_model.columns else ""
    print(movie_id, title, "->", features[:8])

USE_TAG_FEATURES: True
tags_features_en existe: True
Columna usada para tags: tags_features_en
Tags candidatos detectados: 1035
Tags usados como features: 100
Primeros tags usados: ['a_book', 'a_true_story', 'actio', 'ager', 'appi', 'atio', 'atmospheric', 'bare_chested_male', 'based_o', 'bd_r', 'betamax', 'biography', 'blood', 'bori', 'by_character', 'chase', 'cigarette_smoki', 'clv', 'comedy', 'comi', 'crime', 'criterio', 'cult_film', 'd_wife_relatio', 'depe', 'director', 'doctor', 'docume', 'dog', 'drama', 'drugs', 'dship', 'dtrack', 'dvd_video', 'escape', 'ess', 'ew_york_city', 'explosio', 'family', 'family_relatio', 'father_daughter_relatio', 'father_so', 'fidelity', 'fight', 'fire', 'flashback', 'fra', 'fre', 'frie', 'g_of_age', 'gay', 'ger', 'gla', 'gore', 'horror', 'hospital', 'husba', 'ic_film', 'ist', 'ity', 'kid', 'kiss', 'love', 'marriage', 'martial_arts', 'mother_so', 'murder', 'musical', 'ovel', 'ovel_or_book', 'police', 'politics', 'priso', 'psychotro', 'rape', 'relatio',

## 8. Construir Dataset de LightFM

In [8]:
# ConstrucciÃ³n estable del Dataset de LightFM

MAX_USERS = 5000
MAX_ITEMS = 5000
MAX_INTERACTIONS = 50_000
RANDOM_STATE = 42

print("=== ConstrucciÃ³n Dataset LightFM ===", flush=True)

# 1. Mantener siempre las pelÃ­culas valoradas/vistas por el usuario de Trakt
trakt_rows = all_interactions[
    all_interactions["user_id"].astype(str) == "trakt_user"
].copy()
available_movie_ids = set(movies_model["movieId"].astype("int64"))
trakt_rows = trakt_rows[
    trakt_rows["movieId"].astype("int64").isin(available_movie_ids)
].copy()

trakt_movie_ids = set(
    trakt_rows["movieId"]
    .dropna()
    .astype("int64")
)

print("Interacciones Trakt:", len(trakt_rows), flush=True)
print("PelÃ­culas Trakt:", len(trakt_movie_ids), flush=True)

# 2. Seleccionar pelÃ­culas candidatas: populares + pelÃ­culas de Trakt
top_movie_ids = set(
    movies_model
    .sort_values("rating_count", ascending=False)
    .head(MAX_ITEMS)["movieId"]
    .astype("int64")
)

selected_movie_ids = top_movie_ids | trakt_movie_ids

movies_model_lightfm = movies_model[
    movies_model["movieId"].astype("int64").isin(selected_movie_ids)
].copy()

print("PelÃ­culas originales:", len(movies_model), flush=True)
print("PelÃ­culas usadas:", len(movies_model_lightfm), flush=True)

# 3. Filtrar interacciones a las pelÃ­culas seleccionadas
candidate_interactions = all_interactions[
    all_interactions["movieId"].astype("int64").isin(selected_movie_ids)
].copy()

# 4. Seleccionar usuarios con mÃ¡s interacciones, manteniendo siempre trakt_user
user_counts = candidate_interactions["user_id"].astype(str).value_counts()
selected_users = set(user_counts.head(MAX_USERS).index)
selected_users.add("trakt_user")

candidate_interactions = candidate_interactions[
    candidate_interactions["user_id"].astype(str).isin(selected_users)
].copy()

print("Interacciones tras filtrar usuarios/pelÃ­culas:", len(candidate_interactions), flush=True)
print("Usuarios candidatos:", candidate_interactions["user_id"].astype(str).nunique(), flush=True)

# 5. Muestrear interacciones si todavÃ­a son demasiadas
non_trakt_interactions = candidate_interactions[
    candidate_interactions["user_id"].astype(str) != "trakt_user"
].copy()
remaining_budget = max(0, MAX_INTERACTIONS - len(trakt_rows))
if len(non_trakt_interactions) > remaining_budget:
    non_trakt_interactions = non_trakt_interactions.sample(
        n=remaining_budget,
        random_state=RANDOM_STATE
    ).copy()

# 6. ReaÃ±adir interacciones de Trakt por seguridad
candidate_interactions = pd.concat(
    [non_trakt_interactions, trakt_rows],
    ignore_index=True
).drop_duplicates(subset=["user_id", "movieId"])

print("Interacciones finales:", len(candidate_interactions), flush=True)
print("Usuarios finales:", candidate_interactions["user_id"].astype(str).nunique(), flush=True)
print("PelÃ­culas en interacciones:", candidate_interactions["movieId"].nunique(), flush=True)

# 7. Usar IDs string en todo LightFM
candidate_interactions["user_id"] = candidate_interactions["user_id"].astype(str)
candidate_interactions["item_id"] = candidate_interactions["movieId"].astype("int64").astype(str)

movies_model_lightfm["item_id"] = movies_model_lightfm["movieId"].astype("int64").astype(str)

users = candidate_interactions["user_id"].unique()
items = movies_model_lightfm["item_id"].unique()

# 8. Filtrar features al subconjunto de pelÃ­culas usado
valid_items = set(items)

movie_features_lightfm = {
    str(movie_id): features
    for movie_id, features in movie_features.items()
    if str(movie_id) in valid_items
}

print("Features globales:", len(all_item_features), flush=True)
print("PelÃ­culas con features:", len(movie_features_lightfm), flush=True)

# 9. Comprobaciones
missing_interaction_items = set(candidate_interactions["item_id"]) - valid_items
missing_feature_items = set(movie_features_lightfm.keys()) - valid_items

print("Items en interacciones no registrados:", len(missing_interaction_items), flush=True)
print("Items en features no registrados:", len(missing_feature_items), flush=True)
print("NÃƒÂºmero total de item_features:", len(all_item_features), flush=True)

if missing_interaction_items:
    raise ValueError("Hay item_id en candidate_interactions que no estÃ¡n en movies_model_lightfm")

if missing_feature_items:
    raise ValueError("Hay item_id en movie_features_lightfm que no estÃ¡n en movies_model_lightfm")

# 10. Construir Dataset LightFM
dataset = Dataset(
    user_identity_features=True,
    item_identity_features=True
)

dataset.fit(
    users=users,
    items=items,
    item_features=sorted(all_item_features)
)

print("Dataset fit OK", flush=True)

# 11. Construir interacciones SIN list(...)
interaction_tuples = candidate_interactions[
    ["user_id", "item_id"]
].itertuples(index=False, name=None)

interactions, weights = dataset.build_interactions(interaction_tuples)

print("Interactions OK", flush=True)

# 12. Construir item features
item_feature_tuples = (
    (item_id, features)
    for item_id, features in movie_features_lightfm.items()
)

item_features_matrix = dataset.build_item_features(
    item_feature_tuples,
    normalize=True
)

print("Item features OK", flush=True)

print("interactions shape:", interactions.shape, flush=True)
print("weights shape:", weights.shape, flush=True)
print("item_features_matrix shape:", item_features_matrix.shape, flush=True)
print("interactions nnz:", interactions.nnz, flush=True)
print("item_features_matrix nnz:", item_features_matrix.nnz, flush=True)

=== ConstrucciÃ³n Dataset LightFM ===
Interacciones Trakt: 46
PelÃ­culas Trakt: 46
PelÃ­culas originales: 80107
PelÃ­culas usadas: 5000
Interacciones tras filtrar usuarios/pelÃ­culas: 2469711
Usuarios candidatos: 5001
Interacciones finales: 50000
Usuarios finales: 4999
PelÃ­culas en interacciones: 4693
Features globales: 138
PelÃ­culas con features: 5000
Items en interacciones no registrados: 0
Items en features no registrados: 0
NÃƒÂºmero total de item_features: 138
Dataset fit OK
Interactions OK
Item features OK
interactions shape: (4999, 5000)
weights shape: (4999, 5000)
item_features_matrix shape: (5000, 5138)
interactions nnz: 50000
item_features_matrix nnz: 77282


## 9. Entrenar modelo

In [9]:
LOSS = "logistic"
NO_COMPONENTS = 16
LEARNING_RATE = 0.03
EPOCHS = 5
NUM_THREADS = 1
RANDOM_STATE = 42

model = LightFM(
    loss=LOSS,
    no_components=NO_COMPONENTS,
    learning_rate=LEARNING_RATE,
    random_state=RANDOM_STATE,
)

print("Inicio del entrenamiento LightFM", flush=True)
print("LOSS:", LOSS, flush=True)
print("NO_COMPONENTS:", NO_COMPONENTS, flush=True)
print("EPOCHS:", EPOCHS, flush=True)
print("NUM_THREADS:", NUM_THREADS, flush=True)

model.fit(
    interactions,
    item_features=item_features_matrix,
    epochs=EPOCHS,
    num_threads=NUM_THREADS,
    verbose=True,
)

print("Fin del entrenamiento LightFM", flush=True)

Inicio del entrenamiento LightFM
LOSS: logistic
NO_COMPONENTS: 16
EPOCHS: 5
NUM_THREADS: 1
Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Fin del entrenamiento LightFM


## 10. EvaluaciÃ³n

La evaluaciÃ³n actual se calcula sobre las interacciones usadas en entrenamiento, por lo que no debe considerarse una mÃ©trica final. En una fase posterior se implementarÃ¡ train/test split por usuario.

In [10]:
try:
    from lightfm.evaluation import auc_score, precision_at_k

    precision = precision_at_k(
        model,
        interactions,
        item_features=item_features_matrix,
        k=10,
        num_threads=NUM_THREADS,
    ).mean()
    auc = auc_score(
        model,
        interactions,
        item_features=item_features_matrix,
        num_threads=NUM_THREADS,
    ).mean()
    print(f"precision@10: {precision:.4f}")
    print(f"AUC: {auc:.4f}")
except Exception as exc:
    print("Aviso: no se pudo calcular la evaluaciÃ³n automÃ¡tica de LightFM.")
    print(type(exc).__name__, exc)

precision@10: 0.0013
AUC: 0.4415


## 11. FunciÃ³n de recomendaciÃ³n para trakt_user

In [11]:
def recommend_lightfm_for_trakt_user(
    model,
    dataset,
    movies_df,
    item_features_matrix,
    trakt_user_id="trakt_user",
    watched_movie_ids=None,
    rated_movie_ids=None,
    n=20,
):
    user_id_map, _, item_id_map, _ = dataset.mapping()
    if trakt_user_id not in user_id_map:
        raise ValueError(f"No se ha encontrado el usuario {trakt_user_id} en el Dataset de LightFM")

    internal_user_id = user_id_map[trakt_user_id]
    item_ids = np.array(list(item_id_map.keys()))
    internal_item_ids = np.array([item_id_map[item_id] for item_id in item_ids])

    scores = model.predict(
        user_ids=np.repeat(internal_user_id, len(internal_item_ids)),
        item_ids=internal_item_ids,
        item_features=item_features_matrix,
    )

    result = pd.DataFrame({"movieId": item_ids.astype("int64"), "lightfm_score": scores})

    excluded_ids = set()
    if watched_movie_ids is not None:
        excluded_ids.update(int(movie_id) for movie_id in watched_movie_ids if pd.notna(movie_id))
    if rated_movie_ids is not None:
        excluded_ids.update(int(movie_id) for movie_id in rated_movie_ids if pd.notna(movie_id))

    if excluded_ids:
        result = result.loc[~result["movieId"].isin(excluded_ids)].copy()

    metadata_columns = [
        column
        for column in ["movieId", "title", "year", "genres", "rating_mean", "rating_count", "weighted_rating"]
        if column in movies_df.columns
    ]
    movies_metadata = movies_df[metadata_columns].drop_duplicates(subset=["movieId"]).copy()
    movies_metadata["movieId"] = pd.to_numeric(movies_metadata["movieId"], errors="coerce")
    movies_metadata["year"] = pd.to_numeric(movies_metadata["year"], errors="coerce")
    if "genres" in movies_metadata.columns:
        movies_metadata["genres"] = movies_metadata["genres"].fillna("").astype(str).str.strip()
    movies_metadata = movies_metadata.dropna(subset=["movieId", "year"])
    if "genres" in movies_metadata.columns:
        movies_metadata = movies_metadata.loc[
            movies_metadata["genres"].ne("")
            & movies_metadata["genres"].ne("(no genres listed)")
        ].copy()
    movies_metadata["movieId"] = movies_metadata["movieId"].astype("int64")

    recommendations = result.merge(movies_metadata, on="movieId", how="left")
    recommendations = recommendations.dropna(subset=["year"]).copy()
    if "genres" in recommendations.columns:
        recommendations = recommendations.loc[
            recommendations["genres"].notna()
            & recommendations["genres"].astype(str).str.strip().ne("")
            & recommendations["genres"].astype(str).str.strip().ne("(no genres listed)")
        ].copy()
    recommendations = recommendations.sort_values("lightfm_score", ascending=False).head(n)

    def build_explanation(row):
        genres_value = row.get("genres", "")
        genres_text = ""
        if pd.notna(genres_value):
            genres_text = ", ".join(
                genre.strip()
                for genre in str(genres_value).split("|")
                if genre.strip()
            )
        if genres_text:
            return (
                "Recomendada por el modelo LightFM al combinar patrones de usuarios similares "
                f"con features como {genres_text}."
            )
        return "Recomendada por el modelo LightFM al combinar patrones de usuarios similares con features de pelicula."

    recommendations["explanation"] = recommendations.apply(build_explanation, axis=1)

    output_columns = ["movieId", "title", "year", "genres", "rating_mean", "rating_count"]
    if "weighted_rating" in recommendations.columns:
        output_columns.append("weighted_rating")
    output_columns.extend(["lightfm_score", "explanation"])
    for column in output_columns:
        if column not in recommendations.columns:
            recommendations[column] = np.nan

    return recommendations[output_columns].reset_index(drop=True)

## 12. Ejecutar recomendaciones

In [12]:
watched_movie_ids = set()
if not trakt_watched.empty and "movieId" in trakt_watched.columns:
    watched_movie_ids_series = pd.to_numeric(trakt_watched["movieId"], errors="coerce").dropna()
    watched_movie_ids = set(watched_movie_ids_series.astype("int64"))

print("PelÃ­culas ya vistas en Trakt:", len(watched_movie_ids))
print("PelÃ­culas ya valoradas en Trakt:", len(rated_movie_ids))

lightfm_recommendations = recommend_lightfm_for_trakt_user(
    model=model,
    dataset=dataset,
    movies_df=movies_model_lightfm,
    item_features_matrix=item_features_matrix,
    trakt_user_id="trakt_user",
    watched_movie_ids=watched_movie_ids,
    rated_movie_ids=rated_movie_ids,
    n=20,
)

display(lightfm_recommendations)

output_dir = base_dir / "reports" / "resultados"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "lightfm_recommendations.csv"
lightfm_recommendations.to_csv(output_path, index=False)

print("Recomendaciones guardadas en:", output_path)

PelÃ­culas ya vistas en Trakt: 218
PelÃ­culas ya valoradas en Trakt: 123


,movieId,title,year,genres,rating_mean,rating_count,lightfm_score,explanation
0,5267,"Rookie, The (2002)",2002.0,Drama,3.494977,2588,6.103179,Recomendada por el modelo LightFM al combinar ...
1,3534,28 Days (2000),2000.0,Drama,3.072709,5357,6.103129,Recomendada por el modelo LightFM al combinar ...
2,213,Burnt by the Sun (Utomlyonnye solntsem) (1994),1994.0,Drama,4.015547,1994,6.060610,Recomendada por el modelo LightFM al combinar ...
3,47644,Invincible (2006),2006.0,Drama,3.562833,1313,6.001055,Recomendada por el modelo LightFM al combinar ...
4,3980,Men of Honor (2000),2000.0,Drama,3.633777,4657,5.941515,Recomendada por el modelo LightFM al combinar ...
5,4254,Crocodile Dundee in Los Angeles (2001),2001.0,Comedy|Drama,2.221224,2059,5.908168,Recomendada por el modelo LightFM al combinar ...
6,2272,One True Thing (1998),1998.0,Drama,3.481020,922,5.887862,Recomendada por el modelo LightFM al combinar ...
7,1366,"Crucible, The (1996)",1996.0,Drama,3.396560,2151,5.887587,Recomendada por el modelo LightFM al combinar ...
8,6832,Regarding Henry (1991),1991.0,Drama,3.350909,1100,5.887094,Recomendada por el modelo LightFM al combinar ...
9,1054,Get on the Bus (1996),1996.0,Drama,3.489912,793,5.886060,Recomendada por el modelo LightFM al combinar ...


Recomendaciones guardadas en: C:\Users\alexo\Desktop\TFG\reports\resultados\lightfm_recommendations.csv


## Conclusiones

En esta versiÃ³n del experimento LightFM se ha reforzado la calidad de la seÃ±al antes del entrenamiento. Ahora el subconjunto de pelÃ­culas se limpia para eliminar elementos sin `genres` o sin `year`, se mantienen las pelÃ­culas de Trakt aunque no sean populares, se activan las features de identidad de usuario e Ã­tem y se aÃ±aden tags frecuentes con un lÃ­mite controlado.

Activar `item_identity_features=True` ayuda a que LightFM distinga pelÃ­culas concretas y no solo grupos genÃ©ricos como combinaciones de gÃ©nero, dÃ©cada y popularidad. Eso suele reducir empates y hace que el modelo capture mejor preferencias finas de usuario.

Los tags se limitan a `MAX_TAG_FEATURES` para evitar ruido, explosiones de dimensionalidad y features demasiado genÃ©ricas. De esta forma se conservan solo los tags mÃ¡s frecuentes y Ãºtiles.

Aun asÃ­, LightFM sigue siendo menos explicable que el recomendador basado en TF-IDF y perfil semÃ¡ntico. Su valor aquÃ­ es aportar una seÃ±al colaborativa+contenido que puede complementar al sistema explicable.

El siguiente paso serÃ¡ combinar `lightfm_score` con el recomendador explicable en un score hÃ­brido final y usar esa combinaciÃ³n para generar explicaciones mÃ¡s sÃ³lidas.